# PlanDiffusion 评估 (Kaggle)

在 8k 测试集上计算两个指标：
- **Coord-RMSE**：给定真实图结构（A, mask, text），DDPM/DDIM 逆扩散生成坐标，与 GT 坐标对比
- **Type Acc**：给定真实坐标 + 图结构，NodeTypeClassifier 预测节点类型，与 GT 类型对比

**可视化 cell** 展示 4 个阶段：GT 平面图 | 生成坐标（图结构） | 节点类型预测 | 投票渲染平面图

In [ ]:
import os, shutil, sys
REPO_DIR = '/kaggle/working/PlanDiffusion_wzm'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.system(f'git clone -b wzm-shenzhou https://github.com/WeeZHnMin/PlanDiffusion_wzm.git {REPO_DIR}')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('repo ready:', REPO_DIR)

In [ ]:
# ── 中文字体（Kaggle 上只有 wqy-zenhei 可用）────────────────────────────────
import subprocess, matplotlib, matplotlib.font_manager as fm
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-wqy-zenhei'], check=True)
fm.fontManager.addfont('/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc')
matplotlib.rcParams['font.family']        = ['WenQuanYi Zen Hei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print('font:', fm.findfont('WenQuanYi Zen Hei'))

In [ ]:
# ── 路径配置（按实际 Kaggle Dataset 路径修改）────────────────────────────────────
TEST_DATA_PATH = '/kaggle/input/datasets/wzmmmm/plandiffusion-test/test_graph_dataset_8k.jsonl'
VOCAB_DIR      = f'{REPO_DIR}/node_diffusion/unified_vocab_wp'
COMBO_VOCAB    = f'{REPO_DIR}/node_diffusion/type_combo_vocab_old.json'

COORD_CKPT = '/kaggle/input/models/wzmmmm/plandiffusion-weights/pytorch/default/1/coord_model_latest.pt'
TYPE_CKPT  = '/kaggle/input/models/wzmmmm/plandiffusion-weights/pytorch/default/1/type_model_latest.pt'

# ── 评估配置 ──────────────────────────────────────────────────────────────────
EVAL_BATCH = 64
DDIM_STEPS = 0      # 0 = 完整 DDPM 1000 步（论文评估用）；>0 = DDIM 加速（仅快速验证）
N_EVAL     = 0      # 0 = 全部 8000 条

# ── 模型超参 ──────────────────────────────────────────────────────────────────
MODEL_CHANNELS = 384
NUM_LAYERS     = 6
NUM_HEADS      = 6
TIMESTEPS      = 1000
MAX_NODES      = 40
MAX_TEXT_LEN   = 224
N_TYPES        = 33

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import Polygon as ShapelyPolygon  # Kaggle 预装，无需 pip install

# ── 直接从 node_diffusion 模块导入 ────────────────────────────────────────────
from node_diffusion.model      import NodeDiffusionTransformer
from node_diffusion.type_model import NodeTypeClassifier
from node_diffusion.diffusion  import GaussianDiffusion
from node_diffusion.infer      import p_sample_loop
from node_diffusion.render     import (
    find_faces, vote_room_type,
    ROOM_TYPE_ORDER, ROOM_COLORS, ROOM_LABELS,
    load_vocab,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

vocab_cfg      = json.loads(open(os.path.join(VOCAB_DIR, 'vocab_config.json'), encoding='utf-8').read())
BPE_VOCAB_SIZE = vocab_cfg['wp_vocab_size']
print(f'WP vocab size: {BPE_VOCAB_SIZE}')

## 数据集（TestDataset — 测试专用，模块内无对应类）

In [ ]:
class TestDataset(Dataset):
    """从 test JSONL 读取，on-the-fly WordPiece tokenize prompt。"""
    def __init__(self, jsonl_path, tokenizer_path, max_text_len=224, n_max=None):
        self.tokenizer    = Tokenizer.from_file(tokenizer_path)
        self.max_text_len = max_text_len
        self.records      = []
        with open(jsonl_path, encoding='utf-8') as f:
            for line in f:
                self.records.append(json.loads(line.strip()))
                if n_max and len(self.records) >= n_max:
                    break
        print(f'TestDataset: {len(self.records)} samples')

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        x   = torch.from_numpy(np.array(rec['node_coords'], dtype=np.float32).T)  # [2, 40]
        adj = torch.tensor(rec['adj_matrix'],     dtype=torch.float32)             # [40, 40]
        msk = torch.tensor(rec['node_mask'],      dtype=torch.float32)             # [40]
        typ = torch.tensor(rec['node_combo_ids'], dtype=torch.long)                # [40]
        ids = self.tokenizer.encode(rec['prompt']).ids[:self.max_text_len]
        pad = self.max_text_len - len(ids)
        tok = torch.tensor(ids + [0] * pad, dtype=torch.long)
        tkm = torch.tensor([1.] * len(ids) + [0.] * pad, dtype=torch.float32)
        return x, adj, msk, typ, tok, tkm

tokenizer_path = os.path.join(VOCAB_DIR, 'wp_tokenizer.json')
test_ds     = TestDataset(TEST_DATA_PATH, tokenizer_path, MAX_TEXT_LEN, N_EVAL if N_EVAL > 0 else None)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=4, pin_memory=True)

## 加载权重

In [ ]:
def load_ckpt(model, path):
    ck = torch.load(path, map_location=device)
    st = ck['model'] if 'model' in ck else ck
    if any(k.startswith('module.') for k in st):
        st = {k[7:]: v for k, v in st.items()}
    miss, unex = model.load_state_dict(st, strict=False)
    if miss:
        print(f'  missing: {miss}')
    if unex:
        print(f'  unexpected: {unex}')
    print(f'  step={ck.get("step", "?")}')

print('--- coord model')
coord_model = NodeDiffusionTransformer(
    model_channels=MODEL_CHANNELS, num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS, bpe_vocab_size=BPE_VOCAB_SIZE,
).to(device)
load_ckpt(coord_model, COORD_CKPT)
coord_model.eval()

print('--- type model')
type_model = NodeTypeClassifier(
    model_channels=MODEL_CHANNELS, num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS, bpe_vocab_size=BPE_VOCAB_SIZE, n_types=N_TYPES,
).to(device)
load_ckpt(type_model, TYPE_CKPT)
type_model.eval()

diffusion = GaussianDiffusion(TIMESTEPS)
print('Done.')

---
## 快速可视化（挑几条看效果）

每条样本展示 **4 列**：

| 列 | 内容 |
|---|---|
| ① GT | 真实平面图（房间着色 + 标注） |
| ② 生成坐标 | 扩散模型输出的节点坐标（灰色图，只有节点 + 边，无类型） |
| ③ 节点类型 | 同一坐标，节点按预测的主类型着色 |
| ④ 渲染平面图 | 投票算法 → 每个面确定房间类型 → 完整着色渲染 |

In [ ]:
# ── combo vocab（来自 render.load_vocab） ─────────────────────────────────────
id_to_combo = load_vocab(Path(COMBO_VOCAB))   # {int → List[str]}
print(f'Combo vocab: {len(id_to_combo)} types')

# ── 坐标归一化（[0,1] 视口） ─────────────────────────────────────────────────
def make_norm(coords):
    xs = [c[0] for c in coords]; ys = [c[1] for c in coords]
    mnx, mxx = min(xs), max(xs); mny, mxy = min(ys), max(ys)
    span = max(mxx - mnx, mxy - mny, 1.0); mg = span * 0.12
    def norm(x, y): return (x-mnx+mg)/(span+2*mg), (y-mny+mg)/(span+2*mg)
    return norm

# ── Panel 1 & 4：渲染平面图（find_faces + vote_room_type 均来自 render） ──────
def draw_floorplan(ax, coords, adj, node_types, title=''):
    n = len(coords); adj_n = [r[:n] for r in adj[:n]]
    all_nbrs = {i: [j for j in range(n) if i != j and adj_n[i][j] == 1] for i in range(n)}
    faces = find_faces(coords, adj_n)
    norm  = make_norm(coords)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for face in faces:
        rt  = vote_room_type(face, node_types, all_nbrs)
        pts = [norm(*coords[i]) for i in face]
        ax.add_patch(MplPolygon(pts, closed=True,
            facecolor=ROOM_COLORS.get(rt, '#EAEDED'), edgecolor='#555', lw=1.0, alpha=0.88, zorder=1))
        try:    rp = ShapelyPolygon(pts).representative_point(); cx, cy = rp.x, rp.y
        except: cx = sum(p[0] for p in pts)/len(pts); cy = sum(p[1] for p in pts)/len(pts)
        ax.text(cx, cy, ROOM_LABELS.get(rt, rt), ha='center', va='center', fontsize=6.5, color='#222', zorder=3)
    for i in range(n):
        for j in all_nbrs[i]:
            if j > i:
                x0, y0 = norm(*coords[i]); x1, y1 = norm(*coords[j])
                ax.plot([x0, x1], [y0, y1], color='#888', lw=0.7, zorder=2)
    for i in range(n):
        x, y = norm(*coords[i]); ax.plot(x, y, 'o', color='#333', ms=3, zorder=4)
    if title: ax.set_title(title, fontsize=7, pad=3)

# ── Panel 2：纯坐标图（灰色节点 + 边，无类型） ───────────────────────────────
def draw_coord_graph(ax, coords, adj, title=''):
    n = len(coords); adj_n = [r[:n] for r in adj[:n]]
    norm = make_norm(coords)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for i in range(n):
        for j in range(i+1, n):
            if adj_n[i][j] == 1:
                x0, y0 = norm(*coords[i]); x1, y1 = norm(*coords[j])
                ax.plot([x0, x1], [y0, y1], color='#999', lw=1.0, zorder=1)
    for i in range(n):
        x, y = norm(*coords[i]); ax.plot(x, y, 'o', color='#444', ms=5, zorder=2)
    if title: ax.set_title(title, fontsize=7, pad=3)

# ── Panel 3：节点按预测主类型着色 ──────────────────────────────────────────────
def draw_node_types(ax, coords, adj, node_types, title=''):
    """node_types: list of list[str]，取 ROOM_TYPE_ORDER 优先级最高的类型作颜色。"""
    n = len(coords); adj_n = [r[:n] for r in adj[:n]]
    order = {t: k for k, t in enumerate(ROOM_TYPE_ORDER)}
    norm  = make_norm(coords)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for i in range(n):
        for j in range(i+1, n):
            if adj_n[i][j] == 1:
                x0, y0 = norm(*coords[i]); x1, y1 = norm(*coords[j])
                ax.plot([x0, x1], [y0, y1], color='#bbb', lw=0.8, zorder=1)
    for i in range(n):
        primary = min(node_types[i], key=lambda t: order.get(t, len(ROOM_TYPE_ORDER)))
        color   = ROOM_COLORS.get(primary, '#EAEDED')
        x, y    = norm(*coords[i])
        ax.plot(x, y, 'o', color=color, ms=9, zorder=2, markeredgecolor='#333', markeredgewidth=0.5)
        ax.text(x, y, ROOM_LABELS.get(primary, primary)[:3],
                ha='center', va='center', fontsize=4.5, color='#111', zorder=3)
    if title: ax.set_title(title, fontsize=7, pad=3)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  修改这里选择要可视化的样本
VIZ_INDICES    = [0, 1, 2, 3, 4]   # 指定下标
VIZ_RANDOM_N   = 0                  # >0 时随机抽取 N 条（忽略 VIZ_INDICES）
VIZ_DDIM_STEPS = 0                  # 0 = 完整 DDPM 1000 步（与论文评估一致）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import random as _rnd
if VIZ_RANDOM_N > 0:
    VIZ_INDICES = _rnd.sample(range(len(test_ds)), min(VIZ_RANDOM_N, len(test_ds)))
    print(f'随机抽取: {VIZ_INDICES}')

N_ROWS = len(VIZ_INDICES)
fig, axes = plt.subplots(N_ROWS, 4, figsize=(16, N_ROWS * 4 + 0.6))
if N_ROWS == 1: axes = [axes]

for c, ct in enumerate(['① GT 平面图', '② 生成坐标（纯图）', '③ 节点类型预测', '④ 投票渲染平面图']):
    axes[0][c].set_title(ct, fontsize=8, fontweight='bold', pad=14)

for row, idx in enumerate(VIZ_INDICES):
    rec = test_ds.records[idx]
    x_gt, adj_t, mask_t, types_t, tok_t, tkm_t = test_ds[idx]

    x_b   = x_gt.unsqueeze(0).to(device)
    adj_b = adj_t.unsqueeze(0).to(device)
    msk_b = mask_t.unsqueeze(0).to(device)
    tok_b = tok_t.unsqueeze(0).to(device)
    tkm_b = tkm_t.unsqueeze(0).to(device)
    model_kwargs = dict(adj_matrix=adj_b, node_mask=msk_b, prompt_tokens=tok_b, prompt_mask=tkm_b)

    # Stage 2: coord diffusion → 生成节点坐标（完整 DDPM 1000 步）
    with torch.no_grad():
        x_pred = p_sample_loop(coord_model, diffusion, (1, 2, MAX_NODES), model_kwargs, device, VIZ_DDIM_STEPS)

    # Stage 3: type classifier → 预测节点类型
    with torch.no_grad():
        logits = type_model(x_pred, adj_b, msk_b, prompt_tokens=tok_b, prompt_mask=tkm_b)
        pred_type_ids = logits.float().argmax(dim=-1)[0].cpu().tolist()

    n_nodes = int(mask_t.sum().item())
    rmse    = (((x_pred[0,:,:n_nodes] - x_b[0,:,:n_nodes].float())**2).sum(0).mean() / 2).sqrt().item()

    gt_coords   = [(float(c[0]), float(c[1])) for c in rec['node_coords'][:n_nodes]]
    gt_types    = [id_to_combo.get(rec['node_combo_ids'][i], ['other']) for i in range(n_nodes)]
    gt_adj      = rec['adj_matrix']

    pred_coords = [(float(x_pred[0,0,i].item()), float(x_pred[0,1,i].item())) for i in range(n_nodes)]
    pred_types  = [id_to_combo.get(pred_type_ids[i], ['other']) for i in range(n_nodes)]

    prompt_short = (rec['prompt'][:72]+'…') if len(rec['prompt']) > 72 else rec['prompt']

    draw_floorplan  (axes[row][0], gt_coords,   gt_adj, gt_types,   title=f'idx={idx}  n={n_nodes}')
    draw_coord_graph(axes[row][1], pred_coords, gt_adj,             title=f'RMSE={rmse:.2f}')
    draw_node_types (axes[row][2], pred_coords, gt_adj, pred_types, title='predicted types')
    draw_floorplan  (axes[row][3], pred_coords, gt_adj, pred_types, title='voted floor plan')
    axes[row][0].set_xlabel(prompt_short, fontsize=5.5, labelpad=4)

fig.suptitle('PlanDiffusion 推理可视化', fontsize=11, y=1.002)
plt.tight_layout()
plt.savefig('/kaggle/working/viz_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('已保存至 /kaggle/working/viz_pipeline.png')

---
## 评估 1：Coord-RMSE（全量）

In [ ]:
coord_rmse_list = []
nb = len(test_loader)
for bi, (x_gt, adj, msk, types, tok, tkm) in enumerate(test_loader):
    x_gt = x_gt.to(device); adj = adj.to(device); msk = msk.to(device)
    tok  = tok.to(device);   tkm = tkm.to(device)
    B, _, N = x_gt.shape
    kwargs  = dict(adj_matrix=adj, node_mask=msk, prompt_tokens=tok, prompt_mask=tkm)
    xp = p_sample_loop(coord_model, diffusion, (B, 2, N), kwargs, device, DDIM_STEPS)
    sq = ((xp - x_gt.float())**2).sum(dim=1)  # [B, N]
    valid = msk.bool()
    for b in range(B):
        nv = valid[b].sum().item()
        if nv > 0:
            coord_rmse_list.append((sq[b][valid[b]].mean() / 2).sqrt().item())
    if (bi+1) % 10 == 0:
        print(f'  [{bi+1}/{nb}]  mean={np.mean(coord_rmse_list):.4f}')

m, s = np.mean(coord_rmse_list), np.std(coord_rmse_list)
print(f'\n=== Coord-RMSE ===  N={len(coord_rmse_list)}  mean={m:.4f}  std={s:.4f}  median={np.median(coord_rmse_list):.4f}')

## 评估 2：Type Acc（全量）

In [ ]:
total_correct = total_valid = 0
with torch.no_grad():
    for bi, (x_gt, adj, msk, types, tok, tkm) in enumerate(test_loader):
        x_gt  = x_gt.to(device);  adj   = adj.to(device);  msk   = msk.to(device)
        types = types.to(device); tok   = tok.to(device);   tkm   = tkm.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = type_model(x_gt, adj, msk, prompt_tokens=tok, prompt_mask=tkm)
        pred  = logits.float().argmax(dim=-1)
        valid = msk.bool()
        total_correct += (pred.eq(types) & valid).sum().item()
        total_valid   += valid.sum().item()
        if (bi+1) % 10 == 0:
            print(f'  [{bi+1}/{nb}]  acc={total_correct/max(total_valid,1):.4f}')

acc = total_correct / max(total_valid, 1)
print(f'\n=== Type Acc ===  valid={total_valid}  correct={total_correct}  acc={acc:.4f} ({acc*100:.2f}%)')

## 汇总

In [ ]:
mode = f'DDIM-{DDIM_STEPS}步' if DDIM_STEPS > 0 else f'DDPM-{TIMESTEPS}步'
print('='*45)
print(f'PlanDiffusion 评估  N={len(test_ds)}')
print('='*45)
print(f'采样:        {mode}')
print(f'Coord-RMSE:  {m:.4f} ± {s:.4f}')
print(f'Type Acc:    {acc*100:.2f}%')
print('='*45)

with open('/kaggle/working/eval_results.json', 'w') as f:
    json.dump({
        'n_samples': len(test_ds), 'sampling_mode': mode,
        'coord_rmse_mean': round(m, 4), 'coord_rmse_std': round(s, 4),
        'type_acc': round(acc, 4),
    }, f, indent=2)
print('已保存至 /kaggle/working/eval_results.json')